# Version 21: dual-branch ECG classifier with *active* CORAL alignment

**Why v21 exists.** v19's CORAL term used the Deep-CORAL 1/(4d²) normalization on a 192-d feature; its logged value was 3–6e-4, i.e. λ·CORAL ≈ 2e-5 against a focal loss of 0.005–0.1 — the alignment exerted no force and the three "CORAL" candidates were effectively the control. v21 keeps everything else identical and makes the term scale-free: `‖C_s − C_t‖²_F / ‖C_s‖²_F` with λ = 0.5, and prints the CORAL share of the total loss every epoch so the effect is verifiable in the log.

**Evidence this notebook is built on.**
- 31 submissions of inductive S003-family models sit in 0.54–0.60 public macro F1 (best S003 = 0.60437). Run-to-run noise on this leaderboard is ±0.03 (S003 0.604 / S023 0.590 / S029 0.551 for near-identical models), so only large deltas mean anything.
- The test set is 25 full recordings (N-dominated, 17 unseen in train; adversarial train/test AUC ≈ 0.985). Train was class-subsampled (N capped at 45,000, V = 31%). Every failure so far is a *recording-shift* failure, e.g. an inverted-lead test record (`0.932`) labelled ~95% Ventricular by S003-family models.
- V13 rebuilt a friend's 0.70-public architecture (3-channel derivatives, ResNet + attention/avg/max pooling, tabular MLP, late fusion, 5-fold) *without* its training ingredients and scored 0.54–0.59. The ingredients left out — **CORAL alignment to unlabeled test batches, weighted focal loss, class-balanced sampling, shift/gain/noise augmentation, shift TTA** — are therefore the candidate mechanism. CORAL is the only one that addresses the shift itself.

**Rules.** Fresh random init, PyTorch only, official NPPE-2 data only, no test labels. CORAL uses unlabeled *test features* in a loss term; that is permitted by the competition rules (no external data, no test labels, no pretrained weights) but crosses the project's own earlier "test for inference only" policy — S032 is the inductive control so the effect is measured, not assumed. Nothing here submits to Kaggle.

| ID | Candidate | Difference |
|---|---|---|
| S036 | `v21a_coral` | friend recipe + scale-free CORAL λ=0.5, stratified 5-fold ensemble, 3-shift TTA |
| S037 | `v21b_coral_full` | same recipe on full data × 3 seeds at the CV-derived epoch budget |

`AUDIT` = structure only. `SMOKE` = 1 fold, 1 short epoch, no test read (CORAL disabled since no test features). `FINAL_CV` = everything.

In [ ]:
import time, os, json, hashlib, random, copy
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix

NOTEBOOK_START = time.perf_counter()
RUN_MODE = os.environ.get("NPPE2_RUN_MODE", "FINAL_CV")   # AUDIT | SMOKE | FINAL_CV
QUICK = os.environ.get("NPPE2_V21_QUICK", "0") == "1"     # local end-to-end verification only; never set on Kaggle
assert RUN_MODE in {"AUDIT", "SMOKE", "FINAL_CV"}

SEED = 42
N_FOLDS = 2 if QUICK else 5
MAX_EPOCHS, PATIENCE = (1, 1) if QUICK else (25, 4)
FULL_SEEDS = (42,) if QUICK else (42, 142, 242)
LR, WD, BATCH = 1e-3, 1e-4, 256
FOCAL_GAMMA, CORAL_LAMBDA = 2.0, 0.5   # CORAL is scale-free in v21 (relative Frobenius distance), hence the larger λ
SAMPLER_POWER = 0.5        # ponytail: sampling prob ∝ n_c^-0.5 (softened balance); 1.0 = fully balanced
TTA_SHIFTS = (-3, 0, 3)
SOURCE_SNAP = 0.002        # recording-median-RR grid, used only for per-record diagnostics
TEST_PRIOR = np.array([0.89, 0.03, 0.07, 0.01])  # diagnostic-only reweighting

CANDIDATES = {
    "v21a_coral":      {"id": "S036", "coral": True, "polaug": False, "full": False, "file": "nppe2-s036-v21-exp2p-strat5-dual-focal-coralrel.csv"},
    "v21b_coral_full": {"id": "S037", "coral": True, "polaug": False, "full": True,  "file": "nppe2-s037-v21-exp2p-fullseed3-dual-focal-coralrel.csv"},
}
KNOWN_HASHES = {
    "S003": "971f79a383b463c53fc3fef62e1d46e446bdbdfe1a38ac7de091258eae9c852b",
    "S018": "895653dcc8d6e8b2bd863b265180d1cfe61c51045b2beeeadf35ddf9b8d4c04e",
    "S023": "b48d3a739bc1ad40c537663d7ab90a1a3a0c58b8d537f06ff270b71a9c0c7e84",
    "S026": "f89d5a61764100d4e3ec0442bf46ba73433136a9ff3f010e18413adfeaf8ea2d",
    "S027": "1b4e017904d9def50cb967eac7f25f8841a3c93783acd7c862ace52005711b17",
    "S028": "497087d62a07cc7a73c5b96f437cc3171adc86070f04742fbb2d42017c4d4dc5",
    "S029": "8912db42", "S030": "91c4d511", "S031": "ad8ab9cc",   # first-8 prefixes; full hashes in the ledger
}

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
DEV = torch.device(os.environ.get("NPPE2_V21_DEVICE") or ("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")))
OUT = Path("/kaggle/working/experiment2p_v21") if Path("/kaggle/working").is_dir() else Path(os.environ.get("NPPE2_V21_OUTPUT_DIR", "/tmp/experiment2p_v21"))
(OUT / "candidate_submissions").mkdir(parents=True, exist_ok=True); (OUT / "reports").mkdir(exist_ok=True)
print(RUN_MODE, "quick" if QUICK else "full", DEV, torch.__version__, OUT)

## 1. Official data (test is read only in FINAL_CV)

In [ ]:
SLUG = "nppe-2-t-2-26-ecg-heartbeat-arrhythmia-classification"
SIG = [f"sig_{i}" for i in range(250)]; RR = ["pre_rr", "post_rr", "rr_ratio"]
cands = [Path("/kaggle/input/competitions") / SLUG / "nppe2_dataset", Path("/kaggle/input") / SLUG / "nppe2_dataset", Path("/kaggle/input") / SLUG]
if os.environ.get("NPPE2_DATA_DIR"): cands.insert(0, Path(os.environ["NPPE2_DATA_DIR"]))
DATA = next(p for p in cands if (p / "train.csv").is_file())
dt = {c: "float32" for c in SIG + RR}
train = pd.read_csv(DATA / "train.csv", dtype={"id": "string", "label": "int64", **dt})
assert list(train.columns) == ["id", *SIG, *RR, "label"] and train.id.is_unique and set(train.label.unique()) <= set(range(4))
X_TR, R_TR, Y = train[SIG].to_numpy(np.float32), train[RR].to_numpy(np.float32), train.label.to_numpy(np.int64)
OFFICIAL = str(DATA).startswith("/kaggle/")
if OFFICIAL:
    assert train.shape == (71_748, 255) and np.bincount(Y, minlength=4).tolist() == [45_000, 3_599, 22_552, 597]
test = X_TE = R_TE = None
if RUN_MODE == "FINAL_CV":
    test = pd.read_csv(DATA / "test.csv", dtype={"id": "string", **dt})
    sub = pd.read_csv(DATA / "sample_submission.csv", dtype={"id": "string"})
    assert list(test.columns) == ["id", *SIG, *RR] and test.id.is_unique and test.id.equals(sub.id)
    if OFFICIAL: assert test.shape == (68_914, 254)
    X_TE, R_TE = test[SIG].to_numpy(np.float32), test[RR].to_numpy(np.float32)
CLASS_N = np.bincount(Y, minlength=4)
print("train", X_TR.shape, "| class counts", CLASS_N.tolist(), "| test read:", test is not None)

## 2. Features

- **Signal (3 channels):** raw beat, first difference, second difference — each channel standardized per row (fixed, label-free transform).
- **Tabular (18):** RR6 (log1p pre/post/ratio + 3 missing flags), log post/median-RR, log pre/post, and 10 deterministic morphology descriptors — R-peak offset/sign/amplitude, QRS half-max width, QRS area, pre/QRS/post window energies and their ratios. Robust-scaled (median/IQR) with statistics **fitted on the training rows of each model only**.
- The 0.002 s recording-median-RR proxy is used only to report per-record prediction counts in test (e.g. the inverted-lead record `0.932`).

In [ ]:
def groups_of(R):
    m = R[:, 0] / R[:, 2]
    return np.round(np.where(np.isfinite(m), np.round(m / SOURCE_SNAP) * SOURCE_SNAP, -1.0), 3)

def rowz(a): return (a - a.mean(1, keepdims=True)) / (a.std(1, keepdims=True) + 1e-6)

def signal_channels(X):
    d1 = np.diff(X, axis=1, prepend=X[:, :1]); d2 = np.diff(d1, axis=1, prepend=d1[:, :1])
    return np.stack([rowz(X), rowz(d1), rowz(d2)], 1).astype(np.float32)

def raw_tab(X, R):
    pre, post, ratio = R[:, 0], R[:, 1], R[:, 2]
    med = np.where(np.isfinite(ratio) & (ratio > 0), pre / ratio, np.nan)
    with np.errstate(all="ignore"):
        rr = [np.log1p(np.nan_to_num(pre, nan=0.8)), np.log1p(np.nan_to_num(post, nan=0.8)), np.log1p(np.nan_to_num(ratio, nan=1.0)),
              np.isnan(pre), np.isnan(post), np.isnan(ratio),
              np.log(np.clip(np.nan_to_num(post / med, nan=1.0), 0.05, 20)), np.log(np.clip(np.nan_to_num(pre / post, nan=1.0), 0.05, 20))]
    a = np.abs(X); c = a[:, 100:150]; pk = c.argmax(1) + 100; pv = X[np.arange(len(X)), pk]
    qrs = a[:, 110:141]; half = (qrs >= 0.5 * np.abs(pv)[:, None]).sum(1)
    e_pre, e_qrs, e_post = (X[:, :100] ** 2).mean(1), (X[:, 110:141] ** 2).mean(1), (X[:, 150:] ** 2).mean(1)
    morph = [(pk - 125) / 25.0, np.sign(pv), np.abs(pv), half / 31.0, qrs.sum(1) / 31.0,
             np.log1p(e_pre), np.log1p(e_qrs), np.log1p(e_post), np.log((e_pre + 1e-4) / (e_qrs + 1e-4)), np.log((e_post + 1e-4) / (e_qrs + 1e-4))]
    return np.stack([np.asarray(v, np.float32) for v in rr + morph], 1)
SIGN_COLS = [9]   # np.sign(peak) flips under lead inversion; everything else is polarity-invariant

class Scaler:
    def fit(s, t):
        s.med = np.median(t, 0); s.iqr = np.percentile(t, 75, 0) - np.percentile(t, 25, 0); s.iqr[s.iqr < 1e-6] = 1.0; return s
    def transform(s, t): return np.clip((t - s.med) / s.iqr, -10, 10).astype(np.float32)

SIG_TR, TAB_TR = signal_channels(X_TR), raw_tab(X_TR, R_TR)
SIG_TE, TAB_TE = (signal_channels(X_TE), raw_tab(X_TE, R_TE)) if X_TE is not None else (None, None)
assert np.isfinite(SIG_TR).all() and np.isfinite(TAB_TR).all()
SPLITS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(X_TR, Y))
print("signal", SIG_TR.shape, "tab", TAB_TR.shape, "| folds", [np.bincount(Y[va], minlength=4).tolist() for _, va in SPLITS])

## 3. Fresh PyTorch model — dual branch, late fusion

Signal branch: Conv1d ResNet (GroupNorm — BatchNorm statistics do not transfer across recordings) → temporal attention pooling ⊕ average pooling ⊕ max pooling → 128-d. Tabular branch: MLP → 64-d. Late fusion: concat → 192-d fused feature (this is what CORAL aligns) → classifier. Fresh random init on every construction.

In [ ]:
def gn(c): return nn.GroupNorm(8 if c % 8 == 0 else 1, c)
class Block(nn.Module):
    def __init__(s, i, o, k, st=1):
        super().__init__()
        s.m = nn.Sequential(nn.Conv1d(i, o, k, st, k // 2, bias=False), gn(o), nn.GELU(), nn.Dropout(0.05), nn.Conv1d(o, o, k, 1, k // 2, bias=False), gn(o))
        s.sc = nn.Identity() if i == o and st == 1 else nn.Conv1d(i, o, 1, st, bias=False)
    def forward(s, x): return nn.functional.gelu(s.m(x) + s.sc(x))

class Net(nn.Module):
    def __init__(s, cin=3, ntab=18, w=32):
        super().__init__()
        s.enc = nn.Sequential(nn.Conv1d(cin, w, 15, padding=7, bias=False), gn(w), nn.GELU(),
                              Block(w, w, 9), Block(w, w, 9), Block(w, 2*w, 7, 2), Block(2*w, 2*w, 7), Block(2*w, 4*w, 5, 2), Block(4*w, 4*w, 5))
        s.att = nn.Conv1d(4*w, 1, 1)
        s.sig_out = nn.Sequential(nn.Linear(12*w, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2))
        s.tab = nn.Sequential(nn.Linear(ntab, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.1), nn.Linear(64, 64), nn.LayerNorm(64), nn.GELU())
        s.head = nn.Sequential(nn.Linear(192, 128), nn.GELU(), nn.Dropout(0.3), nn.Linear(128, 4))
        for m in s.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)): nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
    def features(s, x, t):
        h = s.enc(nn.functional.pad(x, (3, 3)))                       # [B, 4w, 64]
        a = torch.softmax(s.att(h), dim=2)
        pooled = torch.cat([(h * a).sum(2), h.mean(2), h.amax(2)], 1)  # attention ⊕ avg ⊕ max
        return torch.cat([s.sig_out(pooled), s.tab(t)], 1)             # 192-d fused feature
    def forward(s, x, t): return s.head(s.features(x, t))

print("params:", f"{sum(p.numel() for p in Net().parameters()):,}")

## 4. Losses, augmentation, TTA

- **Weighted focal loss** (γ = 2): class weights ∝ n_c^-0.5, normalized to mean 1 → heavier minority penalty on top of the softened-balance sampler (`SAMPLER_POWER`).
- **CORAL** (λ = 0.5, scale-free): `‖C_s − C_t‖²_F / ‖C_s‖²_F` between the fused-feature covariances of the labelled training batch and a random unlabeled test batch of the same size; the log prints its share of the total loss. Label-free; the test batch contributes no supervision.
- **Augmentation** (in-batch, GPU): temporal shift ±4 samples (zero-padded), gain × e^N(0, 0.1), Gaussian noise σ = 0.02; `polaug` adds a per-sample sign flip of all channels with the peak-sign tabular column flipped consistently.
- **TTA**: probabilities averaged over shifts {−3, 0, +3}.

In [ ]:
CLASS_W = torch.tensor(CLASS_N ** -0.5 / np.mean(CLASS_N ** -0.5), dtype=torch.float32, device=DEV)
def focal_loss(logits, y):
    logp = torch.log_softmax(logits, 1); lp = logp.gather(1, y[:, None]).squeeze(1); p = lp.exp()
    return (-(1 - p) ** FOCAL_GAMMA * lp * CLASS_W[y]).mean()

def coral_loss(fs, ft):
    def cov(f): f = f - f.mean(0, keepdim=True); return f.T @ f / (len(f) - 1)
    cs, ct = cov(fs), cov(ft); return ((cs - ct) ** 2).sum() / ((cs ** 2).sum() + 1e-8)   # scale-free: relative covariance distance

def shift(x, k):
    if k == 0: return x
    return nn.functional.pad(x, (max(k, 0), max(-k, 0)))[:, :, max(-k, 0):x.shape[2] + max(-k, 0)]

def augment(x, t, polaug):
    B, dev = x.shape[0], x.device
    k = int(torch.randint(-4, 5, (1,)))
    x = shift(x, k) * torch.exp(torch.randn(B, 1, 1, device=dev) * 0.10) + 0.02 * torch.randn_like(x)
    if polaug:
        flip = (torch.rand(B, device=dev) < 0.5).float() * (-2.0) + 1.0
        x = x * flip[:, None, None]; t = t.clone(); t[:, SIGN_COLS] = t[:, SIGN_COLS] * flip[:, None]
    return x, t

@torch.no_grad()
def predict(model, sig, tab, bs=2048, tta=TTA_SHIFTS):
    model.eval(); out = []
    for i in range(0, len(sig), bs):
        xs, xt = torch.tensor(sig[i:i+bs]).to(DEV), torch.tensor(tab[i:i+bs]).to(DEV)
        out.append(sum(torch.softmax(model(shift(xs, k), xt), 1) for k in tta).cpu().numpy() / len(tta))
    p = np.concatenate(out); assert np.isfinite(p).all(); return p

def prior_f1(y, pred, prior=TEST_PRIOR):
    cm = confusion_matrix(y, pred, labels=range(4)).astype(float)
    cm = cm / np.maximum(cm.sum(1, keepdims=True), 1) * prior[:, None]
    return 2 * np.diag(cm) / np.maximum(cm.sum(0) + cm.sum(1), 1e-9)

def metrics(y, p):
    pred = p.argmax(1); f = f1_score(y, pred, average=None, labels=range(4), zero_division=0); pf = prior_f1(y, pred)
    return {"macro": round(float(f.mean()), 4), "f1": [round(float(v), 3) for v in f], "prior_macro": round(float(pf.mean()), 4),
            "prior_f1": [round(float(v), 3) for v in pf], "pred_counts": np.bincount(pred, minlength=4).tolist()}

## 5. Training engine

AdamW, ReduceLROnPlateau on validation macro F1, early stopping (patience 4), best-state restore within the same run. Full-data seeds train for a fixed epoch budget (mean best epoch of S033's folds). The class-balanced sampler draws `len(train)` rows per epoch with replacement, probability ∝ n_c^-SAMPLER_POWER.

In [ ]:
def fit(sig, tab, y, seed, cfg, tab_te=None, eval_set=None, fixed_epochs=None):
    seed_all(seed)
    model = Net(sig.shape[1], tab.shape[1]).to(DEV)
    Xs, Xt, Yt = torch.tensor(sig), torch.tensor(tab), torch.tensor(y)
    use_coral = cfg["coral"] and tab_te is not None
    if use_coral: Ts, Tt = torch.tensor(SIG_TE), torch.tensor(tab_te)
    n = len(y); nb = (n + BATCH - 1) // BATCH
    w = torch.tensor((CLASS_N ** -SAMPLER_POWER)[y], dtype=torch.float64); w /= w.sum()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=2)
    best = {"f1": -1.0, "epoch": 0, "state": None}; bad = 0
    epochs = MAX_EPOCHS if eval_set is not None else fixed_epochs
    for ep in range(epochs):
        model.train(); idx_all = torch.multinomial(w, n, replacement=True); tl = tc = 0.0; t0 = time.perf_counter()
        for b in range(nb):
            idx = idx_all[b * BATCH:(b + 1) * BATCH]
            xs, xt, yy = Xs[idx].to(DEV), Xt[idx].to(DEV), Yt[idx].to(DEV)
            xs, xt = augment(xs, xt, cfg["polaug"])
            fs = model.features(xs, xt); loss = focal_loss(model.head(fs), yy)
            if use_coral:
                j = torch.randint(0, len(Ts), (len(idx),))
                xs2, _ = augment(Ts[j].to(DEV), Tt[j].to(DEV), cfg["polaug"])
                c = coral_loss(fs, model.features(xs2, Tt[j].to(DEV))); loss = loss + CORAL_LAMBDA * c; tc += CORAL_LAMBDA * c.item()
            assert torch.isfinite(loss)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step(); tl += loss.item()
        msg = f"  ep{ep+1} loss={tl/nb:.4f} coral_term={tc/nb:.4f} coral_share={tc/max(tl,1e-9):.0%} {time.perf_counter()-t0:.0f}s"
        if eval_set is not None:
            vf1 = f1_score(eval_set[2], predict(model, eval_set[0], eval_set[1], tta=(0,)).argmax(1), average="macro")
            sched.step(vf1); msg += f" val_macro={vf1:.4f}"
            if vf1 > best["f1"] + 1e-4:
                best = {"f1": vf1, "epoch": ep + 1, "state": copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})}; bad = 0
            else:
                bad += 1
                if bad >= PATIENCE: print(msg + " early-stop"); break
        print(msg)
    if eval_set is not None and best["state"] is not None:
        model.load_state_dict(best["state"]); model.to(DEV); return model, best["epoch"]
    return model, epochs

## 6. Stratified 5-fold CV → OOF + fold-model test probabilities

Stratified (not recording-grouped) folds are used deliberately: every fold model sees every training recording, which is what made the friend's fold ensemble transfer (recording-grouped fold ensembles lost 0.05 to full-data training in V13). OOF is therefore optimistic — it is used for early stopping and per-class collapse checks, not as a leaderboard estimate.

In [ ]:
CV, TEST_PROBS, BEST_EPOCHS = {}, {}, {}
if RUN_MODE != "AUDIT":
    FOLDS = [0] if RUN_MODE == "SMOKE" else range(N_FOLDS)
    for name, cfg in CANDIDATES.items():
        if cfg["full"]: continue
        oof = np.zeros((len(Y), 4), np.float32); rows, beps = [], []
        tp = np.zeros((len(X_TE), 4), np.float32) if X_TE is not None else None
        for f in FOLDS:
            tr, va = SPLITS[f]
            if RUN_MODE == "SMOKE": tr = tr[:4096]
            sc = Scaler().fit(TAB_TR[tr]); tab_tr, tab_va = sc.transform(TAB_TR[tr]), sc.transform(TAB_TR[va])
            tab_te = sc.transform(TAB_TE) if TAB_TE is not None else None
            print(f"[{name}] fold {f} counts={np.bincount(Y[va], minlength=4).tolist()}")
            m, be = fit(SIG_TR[tr], tab_tr, Y[tr], SEED + f, cfg, tab_te, (SIG_TR[va], tab_va, Y[va]))
            oof[va] = predict(m, SIG_TR[va], tab_va); rows.append({"fold": int(f), "best_epoch": be, **metrics(Y[va], oof[va])}); beps.append(be)
            if tp is not None: tp += predict(m, SIG_TE, tab_te) / len(FOLDS)
            del m; torch.cuda.empty_cache() if DEV.type == "cuda" else None
        va_all = np.concatenate([SPLITS[f][1] for f in FOLDS])
        CV[name] = {"oof": metrics(Y[va_all], oof[va_all]), "folds": rows}; BEST_EPOCHS[name] = max(3, int(round(float(np.mean(beps)))))
        TEST_PROBS[name] = tp; np.save(OUT / "reports" / f"oof_{name}.npy", oof)
        print(f"==> {name} OOF {json.dumps(CV[name]['oof'])} epoch budget {BEST_EPOCHS[name]}")
    json.dump({"cv": CV, "epoch_budget": BEST_EPOCHS}, open(OUT / "reports" / "cv_metrics.json", "w"), indent=2)
    display(pd.DataFrame([{"candidate": k, **v["oof"]} for k, v in CV.items()])) if "display" in dir() else print(CV)

## 7. Full-data seeds (S035) → candidate CSVs (no submission is made here)

Every CSV is audited: 68,914 rows, exact `id,label` header, sample-submission ID order, labels 0–3, SHA-256 distinct from every known submission and from each other. Per-record prediction counts are written to the manifest so the inverted-lead record `0.932` can be checked before anything is submitted.

In [ ]:
def sha256(path):
    h = hashlib.sha256(); h.update(Path(path).read_bytes()); return h.hexdigest()

MANIFEST = []
if RUN_MODE == "FINAL_CV":
    G_TE = groups_of(R_TE)
    for name, cfg in CANDIDATES.items():
        if cfg["full"]:
            src = "v21a_coral"; ep = BEST_EPOCHS[src]; probs = np.zeros((len(X_TE), 4), np.float32)
            sc = Scaler().fit(TAB_TR); tab_tr, tab_te = sc.transform(TAB_TR), sc.transform(TAB_TE)
            for s in FULL_SEEDS:
                print(f"[{name}] full-data seed {s} epochs={ep}")
                m, _ = fit(SIG_TR, tab_tr, Y, s, cfg, tab_te, eval_set=None, fixed_epochs=ep)
                probs += predict(m, SIG_TE, tab_te) / len(FULL_SEEDS); del m; torch.cuda.empty_cache() if DEV.type == "cuda" else None
        else:
            probs = TEST_PROBS[name]
        np.save(OUT / "reports" / f"test_probs_{name}.npy", probs)
        pred = probs.argmax(1)
        out = pd.DataFrame({"id": test.id, "label": pred})
        assert len(out) == len(test) and out.id.equals(sub.id) and set(out.label.unique()) <= set(range(4))
        path = OUT / "candidate_submissions" / cfg["file"]; out.to_csv(path, index=False)
        assert open(path).readline().strip() == "id,label"
        h = sha256(path)
        assert not any(h.startswith(k) for k in KNOWN_HASHES.values()), "candidate identical to a known submission"
        assert all(h != r["sha256"] for r in MANIFEST), "candidates not mutually distinct"
        row = {"submission_id": cfg["id"], "candidate": name, "file": cfg["file"], "sha256": h,
               "pred_counts": np.bincount(pred, minlength=4).tolist(),
               "per_record_pred": {str(g): np.bincount(pred[G_TE == g], minlength=4).tolist() for g in np.unique(G_TE)},
               "kaggle_description": f"{cfg['id']} | v21 exp2p | {'fullseed3' if cfg['full'] else 'strat5'} | {name} | sha={h[:8]}"}
        MANIFEST.append(row); print(json.dumps({k: v for k, v in row.items() if k != "per_record_pred"}))
        print("   record 0.932 (inverted lead):", row["per_record_pred"].get("0.932"))
    json.dump(MANIFEST, open(OUT / "reports" / "candidate_manifest.json", "w"), indent=2)

## 8. Compliance summary

In [ ]:
print(json.dumps({
    "run_mode": RUN_MODE, "quick_local_verification": QUICK, "device": str(DEV),
    "runtime_min": round((time.perf_counter() - NOTEBOOK_START) / 60, 1),
    "training_data": "official NPPE-2 train.csv only; unlabeled test.csv features enter only the CORAL covariance term (no test labels, no pseudo-labels)",
    "weights": "fresh random init per model; no pretrained/external checkpoints; best-state restore only within the same run",
    "preprocessing": "fixed label-free transforms; tabular robust scaler fitted on each model's training rows only",
    "decision_rule": "argmax of TTA/fold/seed-averaged probabilities; no thresholding, priors, or count tuning",
    "kaggle_submission_api_called": False,
    "candidates": [r["kaggle_description"] for r in MANIFEST],
}, indent=2))